In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

# Define the base path
base_path = '/content/drive/MyDrive/02_NandiSeedRecommender2'

if os.path.exists(base_path):
    print(f"--- FILE STRUCTURE FOR: {base_path} ---\n")
    for root, dirs, files in os.walk(base_path):
        # Calculate depth for pretty printing
        level = root.replace(base_path, '').count(os.sep)
        indent = ' ' * 4 * level
        print(f"{indent}{os.path.basename(root)}/")

        sub_indent = ' ' * 4 * (level + 1)
        for f in sorted(files):
            print(f"{sub_indent}{f}")
else:
    print(f"Error: The path {base_path} does not exist. Please check your Drive mounting.")

--- FILE STRUCTURE FOR: /content/drive/MyDrive/02_NandiSeedRecommender2 ---

02_NandiSeedRecommender2/
    DataExtractionCode.ipynb
    Nandi_Seed_Recommender___Methodology__Rough_.pdf
    SeedRecommenderCode.ipynb
    SuitabilityCodev5.ipynb
    NandiCounties/
        counties.dbf
        counties.prj
        counties.shp
        counties.shx
    NandiWards/
        Copy of ward.results.formatted.dbf
        Copy of ward.results.formatted.index
        Copy of ward.results.formatted.prj
        Copy of ward.results.formatted.shp
        Copy of ward.results.formatted.shx
    Soil_Data/
        iSDA_Aluminum_0-20cm.tif
        iSDA_Bedrock_Depth_0-200cm.tif
        iSDA_Bulk_Density_0-20cm.tif
        iSDA_Bulk_Density_20-50cm.tif
        iSDA_Calcium_0-20cm.tif
        iSDA_Calcium_20-50cm.tif
        iSDA_Clay_Content_0-20cm.tif
        iSDA_Clay_Content_20-50cm.tif
        iSDA_Iron_0-20cm.tif
        iSDA_Iron_20-50cm.tif
        iSDA_Magnesium_0-20cm.tif
        iSDA_Magnesium_20-

In [ ]:
import os
import rasterio
import numpy as np

def diagnose_nandi_files(lat, lon, season='LongRains'):
    FINAL_DIR = '/content/drive/MyDrive/02_NandiSeedRecommender2/Final_Outputs'
    RAW_DIR = os.path.join(FINAL_DIR, f'Raw_Values_{season}')
    FACTORS_DIR = os.path.join(FINAL_DIR, f'Factors_{season}')

    # List of critical files to check for N/A issues
    to_check = [
        ('Elevation', os.path.join(FINAL_DIR, 'Nandi_Elevation_30m.tif')),
        ('Rain Raw', os.path.join(RAW_DIR, 'mean_season_rain_raw.tif')),
        ('Temp Raw', os.path.join(RAW_DIR, 'mean_temp_raw.tif')),
        ('CEC Apparent Raw', os.path.join(RAW_DIR, 'cec_apparent_raw.tif')),
        ('Temp Factor Score', os.path.join(FACTORS_DIR, 'temp_mean.tif')),
        ('Texture Raw', os.path.join(RAW_DIR, 'texture_raw.tif'))
    ]

    print(f"{'FILE TYPE':<20} | {'EXISTS':<8} | {'PIXEL VALUE':<12} | {'STATUS'}")
    print("-" * 70)

    for label, path in to_check:
        exists = os.path.exists(path)
        val_str = "N/A"
        status = "File Missing"

        if exists:
            try:
                with rasterio.open(path) as src:
                    row, col = src.index(lon, lat)
                    if 0 <= row < src.height and 0 <= col < src.width:
                        val = src.read(1)[row, col]
                        val_str = f"{val:.4f}"
                        if np.isnan(val):
                            status = "NaN (Pixel Empty)"
                        elif val < -900:
                            status = "NoData Mask"
                        else:
                            status = "Valid"
                    else:
                        status = "Out of Bounds"
            except Exception as e:
                status = f"Read Error: {str(e)}"

        print(f"{label:<20} | {str(exists):<8} | {val_str:<12} | {status}")

    # Inspect the JSON
    json_path = os.path.join(FINAL_DIR, 'County_Averages.json')
    if os.path.exists(json_path):
        with open(json_path, 'r') as f:
            data = json.load(f).get(season, {})
            print(f"\nJSON Check for {season}:")
            print(f" - Raw keys found: {list(data.get('raw', {}).keys())}")
    else:
        print("\nJSON Error: County_Averages.json not found.")

# Execute Diagnosis
diagnose_nandi_files(0.2, 35.30)

FILE TYPE            | EXISTS   | PIXEL VALUE  | STATUS
----------------------------------------------------------------------
Elevation            | True     | 2096.0000    | Valid
Rain Raw             | True     | 646.0090     | Valid
Temp Raw             | True     | nan          | NaN (Pixel Empty)
CEC Apparent Raw     | True     | 13.7934      | Valid
Temp Factor Score    | True     | nan          | NaN (Pixel Empty)
Texture Raw          | False    | N/A          | File Missing


NameError: name 'json' is not defined

In [ ]:
# ==============================================================================
# NANDI PRECISION AGRICULTURE: STANDALONE RECOMMENDER SYSTEM (OFFLINE VERSION)
# ==============================================================================
import os
import json
import numpy as np
import pandas as pd
import rasterio
from google.colab import drive

# --- 1. MOUNT GOOGLE DRIVE ---
drive.mount('/content/drive')

# --- 2. DEFINE SYSTEM PATHS ---
BASE = '/content/drive/MyDrive/02_NandiSeedRecommender2'
SEED_XLSX_PATH = os.path.join(BASE, 'Seed_Data/KenyaSeedWebScrape.xlsx')

# --- 3. LOAD EXCEL SEED DATABASE ---
if os.path.exists(SEED_XLSX_PATH):
    seed_df = pd.read_excel(SEED_XLSX_PATH, engine='openpyxl')
    seed_df.columns = seed_df.columns.str.strip()
else:
    seed_df = pd.DataFrame()

# --- 4. SEED RANKING LOGIC ---
def rank_seed_varieties(seed_df, elevation, precip, stress_types):
    if seed_df.empty: return pd.DataFrame(columns=['Variety', 'Note'])
    df = seed_df.copy()
    def check_requirements(row):
        issues = []
        try:
            if not (row["Elevation Min"] <= elevation <= row["Elevation Max"]):
                issues.append("Elevation")
            if not (row["Precipitation Min"] <= precip <= row["Precipitation Max"]):
                issues.append("Rainfall")
            if "dry" in stress_types and row.get("Moisture-Stress Tolerant") != 1:
                issues.append("Not drought-tolerant")
            if "heat" in stress_types and row.get("Heat Tolerant") != 1:
                issues.append("Not heat-tolerant")
            if "cold" in stress_types and row.get("Cold Tolerant") != 1:
                issues.append("Not cold-tolerant")
        except: return "Data formatting error"
        return ", ".join(issues) if issues else "Meets all requirements"

    df["RequirementNotes"] = df.apply(check_requirements, axis=1)
    df["YieldScore"] = pd.to_numeric(df["Potential yield (t/Ha)"], errors="coerce")
    df['PerfectMatch'] = df['RequirementNotes'] == "Meets all requirements"
    df = df.sort_values(["PerfectMatch", "YieldScore"], ascending=[False, False])
    return df.head(5)

# --- 5. THE MASTER REPORT FUNCTION ---
def get_comprehensive_nandi_report(lat, lon, seed_df, season='LongRains'):
    FINAL_DIR = os.path.join(BASE, 'Final_Outputs')
    FACTORS_DIR = os.path.join(FINAL_DIR, f'Factors_{season}')
    RAW_DIR = os.path.join(FINAL_DIR, f'Raw_Values_{season}')

    # Units for display
    UNITS = {
        'ph': '', 'total_nitrogen': '%', 'phosphorus': 'mg/kg', 'potassium': 'cmol/kg',
        'calcium': 'cmol/kg', 'magnesium': 'cmol/kg', 'organic_carbon': '%',
        'ecec': 'cmol/kg', 'zinc': 'mg/kg', 'iron': 'mg/kg', 'clay_content': '%',
        'slope': '%', 'rain': 'mm', 'temp': '°C', 'rh': '%', 'bedrock_depth': 'cm',
        'elevation': 'm', 'stone_content': '%', 'cec_apparent': 'cmol/kg', 'texture': '',
        'min_temp': '°C', 'max_temp': '°C', 'germin_temp': '°C',
        'rh_dev': '%', 'rh_mat': '%',
        'prec_month1': 'mm', 'prec_month2': 'mm', 'prec_month3': 'mm', 'prec_month4': 'mm'
    }

    # iSDA texture class integer → name                                      # ADD
    TEXTURE_NAMES = {                                                         # ADD
        1:  'Clay',                                                           # ADD
        2:  'Silty Clay',                                                     # ADD
        3:  'Silty Clay Loam',                                                # ADD
        4:  'Sandy Clay',                                                     # ADD
        5:  'Sandy Clay Loam',                                                # ADD
        6:  'Clay Loam',                                                      # ADD
        7:  'Silt',                                                           # ADD
        8:  'Silt Loam',                                                      # ADD
        9:  'Loam',                                                           # ADD
        10: 'Sandy Loam',                                                     # ADD
        11: 'Loamy Sand',                                                     # ADD
        12: 'Sand',                                                           # ADD
    }                                                                         # ADD

    # Load Context
    json_path = os.path.join(FINAL_DIR, 'County_Averages.json')
    county_ref = {'scores': {}, 'raw': {}}
    if os.path.exists(json_path):
        with open(json_path, 'r') as f:
            county_ref = json.load(f).get(season, {})

    # Extract Suitability
    suit_path = os.path.join(FINAL_DIR, f'Suit_Mean_{season}.tif')
    with rasterio.open(suit_path) as src:
        row, col = src.index(lon, lat)
        suit_mu = src.read(1)[row, col]
        if np.isnan(suit_mu): return "Coordinates outside Nandi mask."
    with rasterio.open(os.path.join(FINAL_DIR, f'Suit_Std_{season}.tif')) as src:
        suit_std = src.read(1)[row, col]

    def get_raw_val(var):
        lookup = var
        if var == 'rain': lookup = 'mean_season_rain'
        if var == 'temp': lookup = 'mean_temp'
        if var == 'rh':   lookup = 'rh_dev'
        path = os.path.join(RAW_DIR, f'{lookup}_raw.tif')
        if not os.path.exists(path) and var == 'elevation':
            path = os.path.join(FINAL_DIR, 'Nandi_Elevation_30m.tif')
        if os.path.exists(path):
            with rasterio.open(path) as s:
                r, c = s.index(lon, lat)
                return s.read(1)[r, c]
        return np.nan

    elev_val   = get_raw_val('elevation')
    precip_val = get_raw_val('rain')
    temp_val   = get_raw_val('temp')

    stress_codes = []
    if not np.isnan(temp_val):           # ADD: guard against NaN temp
        if temp_val < 16: stress_codes.append("cold")
        if temp_val > 30: stress_codes.append("heat")
    if not np.isnan(precip_val):         # ADD: guard against NaN precip
        if precip_val < 450: stress_codes.append("dry")

    recommendations = rank_seed_varieties(seed_df, elevation=elev_val, precip=precip_val, stress_types=stress_codes)

    # Define Section Variables
    soil_vars = {'ph', 'total_nitrogen', 'phosphorus', 'potassium', 'calcium', 'magnesium',
                 'organic_carbon', 'ecec', 'cec_apparent', 'zinc', 'iron', 'clay_content'}

    climate_vars = {'rain', 'temp', 'rh', 'min_temp', 'max_temp', 'germin_temp',
                    'rh_dev', 'rh_mat', 'prec_month1', 'prec_month2', 'prec_month3', 'prec_month4'}

    # Mirrors get_raw_val remapping for county avg JSON lookup
    raw_key_map = {
        'rain': 'mean_season_rain',
        'temp': 'mean_temp',
        'rh':   'rh_dev',
    }

    soil_list, climate_list, phys_list = [], [], []

    factor_files = sorted([f for f in os.listdir(FACTORS_DIR) if f.endswith('_mean.tif')])

    for f_name in factor_files:
        var = f_name.replace('_mean.tif', '')

        if var.startswith('prob_'): continue  # already shown in risks section

        with rasterio.open(os.path.join(FACTORS_DIR, f_name)) as m_f, \
             rasterio.open(os.path.join(FACTORS_DIR, f'{var}_std.tif')) as s_f:
            r, c = m_f.index(lon, lat)
            score, unc = m_f.read(1)[r, c], s_f.read(1)[r, c]

        if var == 'texture':                                                  # ADD: texture special case
            raw_cls = get_raw_val('texture')                                  # ADD
            if not np.isnan(raw_cls):                                         # ADD
                actual_name = TEXTURE_NAMES.get(int(round(raw_cls)), 'Unknown')  # ADD
            else:                                                             # ADD
                actual_name = 'N/A'                                           # ADD
            avg_cls_raw = county_ref.get('raw', {}).get('texture', np.nan)   # ADD
            if avg_cls_raw and not np.isnan(float(avg_cls_raw)):              # ADD
                avg_name = TEXTURE_NAMES.get(int(round(float(avg_cls_raw))), 'Unknown')  # ADD
            else:                                                             # ADD
                avg_name = 'N/A'                                              # ADD
            entry = {'var': var, 'score': score, 'unc': unc,                 # ADD
                     'actual': actual_name, 'avg_raw': avg_name, 'unit': ''}  # ADD
        else:                                                                 # ADD
            json_key = raw_key_map.get(var, var)
            entry = {'var': var, 'score': score, 'unc': unc, 'actual': get_raw_val(var),
                     'avg_raw': county_ref.get('raw', {}).get(json_key, 0), 'unit': UNITS.get(var, '')}

        if var in soil_vars: soil_list.append(entry)
        elif var in climate_vars: climate_list.append(entry)
        else: phys_list.append(entry)

    soil_list.sort(key=lambda x: x['score'])
    climate_list.sort(key=lambda x: x['score'])
    phys_list.sort(key=lambda x: x['score'])

    # Risks
    risk_labels = ['Cold Risk', 'Heat Risk', 'Drought Risk', 'Flood Risk', 'Overall Failure']
    risk_key_map = {
        'Cold Risk':       'prob_cold',
        'Heat Risk':       'prob_heat',
        'Drought Risk':    'prob_drought',
        'Flood Risk':      'prob_flood',
        'Overall Failure': 'prob_overall_fail'
    }
    risk_data = []
    with rasterio.open(os.path.join(FINAL_DIR, f'Risks_Mean_{season}.tif')) as m_f, \
         rasterio.open(os.path.join(FINAL_DIR, f'Risks_Std_{season}.tif')) as s_f:
        mu_r = m_f.read(window=((row, row+1), (col, col+1)))[:, 0, 0]
        st_r = s_f.read(window=((row, row+1), (col, col+1)))[:, 0, 0]
        for i, label in enumerate(risk_labels):
            risk_data.append({'label': label, 'mu': mu_r[i], 'std': st_r[i],
                              'avg': county_ref.get('scores', {}).get(risk_key_map[label], 0)})

    # --- PRINTING ---
    GREY  = '\033[48;5;250m\033[38;5;16m'
    RESET = '\033[0m'
    W = 115
    print(f"\n{'='*W}\n NANDI PRECISION AGRICULTURE: COMPREHENSIVE REPORT | {season}")
    print(f" GPS: {lat}, {lon} | Elevation: {elev_val:.0f}m | Overall Suitability: {suit_mu*100:.1f}% (±{suit_std:.3f})\n{'='*W}")

    print(f"\n{'TOP SEED RECOMMENDATIONS':<50}\n" + "-" * W)
    print(recommendations[['Variety', 'Potential yield (t/Ha)', 'RequirementNotes']].to_string(index=False))

    header = f"{'VARIABLE':<20} | {'MEASURED':<18} | {'COUNTY AVG':<18} | {'SCORE':<8} | {'UNCERTAINTY'}"
    print(f"\n{header}\n" + "-" * W + "\nCLIMATE RISKS (10-Year Probabilities)")
    for r in risk_data:
        print(f"{r['label']:<20} | {r['mu']*100:>17.1f}% | {r['avg']*100:>17.1f}% | {r['mu']:>8.3f} | ±{r['std']:.3f}")

    def print_section(title, data_list):
        if not data_list: return
        print(f"\n{title}")
        for i, s in enumerate(data_list):
            if isinstance(s['actual'], str):                                  # ADD: texture has string values
                act_str = s['actual']                                         # ADD
                avg_str = s['avg_raw'] if isinstance(s['avg_raw'], str) else 'N/A'  # ADD
            else:
                act_str = f"{s['actual']:>8.2f} {s['unit']}".strip() if not np.isnan(s['actual']) else "N/A"
                avg_str = f"{s['avg_raw']:>8.2f} {s['unit']}".strip() if s['avg_raw'] > 0 else "N/A"
            row_str = f"{s['var']:<20} | {act_str:<18} | {avg_str:<18} | {s['score']:>8.3f} | ±{s['unc']:.3f}"
            if i < 3: print(f"{GREY}{row_str}{RESET}")
            else: print(row_str)

    print_section("SOIL NUTRIENTS & PROPERTIES", soil_list)
    print_section("DETAILED CLIMATE STAGES", climate_list)
    print_section("PHYSICAL & TOPOGRAPHY", phys_list)
    print(f"{'='*W}\n")

# --- EXECUTE ---
get_comprehensive_nandi_report(0.2, 35.30, seed_df, season='LongRains')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

 NANDI PRECISION AGRICULTURE: COMPREHENSIVE REPORT | LongRains
 GPS: 0.2, 35.3 | Elevation: 2096m | Overall Suitability: 70.5% (±0.000)

TOP SEED RECOMMENDATIONS                          
-------------------------------------------------------------------------------------------------------------------
Variety  Potential yield (t/Ha) RequirementNotes
  H6218                   12.32         Rainfall
  H6213                   11.88         Rainfall
  H9401                   11.00         Rainfall
  H6210                   11.00         Rainfall
   H629                   10.56         Rainfall

VARIABLE             | MEASURED           | COUNTY AVG         | SCORE    | UNCERTAINTY
-------------------------------------------------------------------------------------------------------------------
CLIMATE RISKS (10-Year Probabilities)
Cold Risk            |       

In [ ]:
import rasterio
import numpy as np

RAW_LR = '/content/drive/MyDrive/02_NandiSeedRecommender2/Final_Outputs/Raw_Values_LongRains'
lat, lon = 0.2, 35.30

for var in ['min_temp', 'max_temp', 'germin_temp', 'mean_temp']:
    path = f'{RAW_LR}/{var}_raw.tif'
    with rasterio.open(path) as src:
        r, c = src.index(lon, lat)
        data = src.read(1)
        val = data[r, c]

        # Check a 5x5 neighbourhood
        h, w = data.shape
        r0, r1 = max(r-2, 0), min(r+3, h)
        c0, c1 = max(c-2, 0), min(c+3, w)
        patch = data[r0:r1, c0:c1]

        print(f"\n{var}:")
        print(f"  Pixel at ({r},{c}): {val}")
        print(f"  5x5 neighbourhood:\n{patch}")
        print(f"  Raster shape: {data.shape}")
        print(f"  Overall NaN count: {np.isnan(data).sum()} / {data.size}")
        print(f"  Overall value range: {np.nanmin(data):.2f} to {np.nanmax(data):.2f}")


min_temp:
  Pixel at (1337,2076): nan
  5x5 neighbourhood:
[[nan nan nan nan nan]
 [nan nan nan nan nan]
 [nan nan nan nan nan]
 [nan nan nan nan nan]
 [nan nan nan nan nan]]
  Raster shape: (2488, 2593)
  Overall NaN count: 4840793 / 6451384
  Overall value range: 10.36 to 15.94

max_temp:
  Pixel at (1337,2076): nan
  5x5 neighbourhood:
[[nan nan nan nan nan]
 [nan nan nan nan nan]
 [nan nan nan nan nan]
 [nan nan nan nan nan]
 [nan nan nan nan nan]]
  Raster shape: (2488, 2593)
  Overall NaN count: 4840793 / 6451384
  Overall value range: 19.99 to 25.17

germin_temp:
  Pixel at (1337,2076): nan
  5x5 neighbourhood:
[[nan nan nan nan nan]
 [nan nan nan nan nan]
 [nan nan nan nan nan]
 [nan nan nan nan nan]
 [nan nan nan nan nan]]
  Raster shape: (2488, 2593)
  Overall NaN count: 4840793 / 6451384
  Overall value range: 15.61 to 20.95

mean_temp:
  Pixel at (1337,2076): nan
  5x5 neighbourhood:
[[nan nan nan nan nan]
 [nan nan nan nan nan]
 [nan nan nan nan nan]
 [nan nan nan nan nan

In [ ]:
import rasterio
import numpy as np

RAW_LR = '/content/drive/MyDrive/02_NandiSeedRecommender2/Final_Outputs/Raw_Values_LongRains'
lat, lon = 0.2, 35.30

with rasterio.open(f'{RAW_LR}/min_temp_raw.tif') as src:
    r, c = src.index(lon, lat)
    data = src.read(1)
    h, w = data.shape

    # Find nearest valid pixel by expanding radius
    for radius in range(1, 200):
        r0, r1 = max(r-radius, 0), min(r+radius+1, h)
        c0, c1 = max(c-radius, 0), min(c+radius+1, w)
        patch = data[r0:r1, c0:c1]
        valid = patch[np.isfinite(patch)]
        if len(valid) > 0:
            print(f"First valid pixels found at radius={radius}px")
            print(f"  Count of valid pixels in patch: {len(valid)}")
            print(f"  Values: {valid[:5]}")
            break

    # Also check: is the suitability raster valid here?
    FINAL_DIR = '/content/drive/MyDrive/02_NandiSeedRecommender2/Final_Outputs'
    with rasterio.open(f'{FINAL_DIR}/Suit_Mean_LongRains.tif') as s:
        sr, sc = s.index(lon, lat)
        print(f"\nSuitability at same coord: {s.read(1)[sr, sc]}")
        print(f"Suitability pixel: ({sr}, {sc})")
        print(f"Min_temp pixel:    ({r}, {c})")

First valid pixels found at radius=61px
  Count of valid pixels in patch: 123
  Values: [12.087309  12.123464  12.1401205 12.130776  12.1279335]

Suitability at same coord: 0.7054578065872192
Suitability pixel: (1337, 2076)
Min_temp pixel:    (1337, 2076)


In [ ]:
import rasterio
import numpy as np

FINAL_DIR = '/content/drive/MyDrive/02_NandiSeedRecommender2/Final_Outputs'
RAW_LR    = f'{FINAL_DIR}/Raw_Values_LongRains'
lat, lon  = 0.2, 35.30

# Check elevation at the exact same pixel
with rasterio.open(f'{FINAL_DIR}/Nandi_Elevation_30m.tif') as src:
    r, c = src.index(lon, lat)
    elev = src.read(1)
    print(f"Elevation at ({r},{c}): {elev[r,c]}")
    print(f"Elevation 5x5 neighbourhood:\n{elev[max(r-2,0):r+3, max(c-2,0):c+3]}")

# Check mean_season_rain (should be fine — not lapse corrected)
with rasterio.open(f'{RAW_LR}/mean_season_rain_raw.tif') as src:
    r, c = src.index(lon, lat)
    data = src.read(1)
    print(f"\nmean_season_rain at ({r},{c}): {data[r,c]}")
    print(f"NaN count: {np.isnan(data).sum()} / {data.size}")

# Check mean_temp (lapse corrected)
with rasterio.open(f'{RAW_LR}/mean_temp_raw.tif') as src:
    r, c = src.index(lon, lat)
    data = src.read(1)
    print(f"\nmean_temp at ({r},{c}): {data[r,c]}")
    print(f"NaN count: {np.isnan(data).sum()} / {data.size}")
    # Find NaN pattern — is it spatially coherent?
    nan_mask = np.isnan(data)
    print(f"\nNaN pattern around pixel (10x10):")
    print(nan_mask[max(r-5,0):r+5, max(c-5,0):c+5].astype(int))

Elevation at (1337,2076): 2096.0
Elevation 5x5 neighbourhood:
[[2098. 2097. 2097. 2098. 2099.]
 [2098. 2097. 2097. 2098. 2099.]
 [2095. 2095. 2096. 2096. 2095.]
 [2091. 2092. 2095. 2094. 2093.]
 [2091. 2091. 2094. 2094. 2094.]]

mean_season_rain at (1337,2076): 646.009033203125
NaN count: 3359405 / 6451384

mean_temp at (1337,2076): nan
NaN count: 4840793 / 6451384

NaN pattern around pixel (10x10):
[[1 1 1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1 1 1]]


In [ ]:
import rasterio
import numpy as np
from scipy.ndimage import zoom

FINAL_DIR = '/content/drive/MyDrive/02_NandiSeedRecommender2/Final_Outputs'
DIR_CLIMATE = '/content/drive/MyDrive/02_NandiSeedRecommender2/Climate_Data'
lat, lon = 0.2, 35.30

# Load elevation
with rasterio.open(f'{FINAL_DIR}/Nandi_Elevation_30m.tif') as src:
    elevation = src.read(1).astype(np.float32)
    r, c = src.index(lon, lat)

# Load a coarse temp raster to get its shape
with rasterio.open(f'{DIR_CLIMATE}/MeanTemp_LongRains.tif') as src:
    arr = src.read(1).astype(np.float32)
    coarse_shape = arr.shape
    print(f"Coarse temp shape: {coarse_shape}")
    print(f"30m elevation shape: {elevation.shape}")

# Replicate the double-zoom e_ref calculation
H, W = elevation.shape
z_r = coarse_shape[0] / H
z_c = coarse_shape[1] / W
elev_coarse = zoom(elevation, (z_r, z_c), order=1)
print(f"\nelev_coarse at approx location: {elev_coarse[int(r*z_r), int(c*z_c)]}")
print(f"elev_coarse NaN count: {np.isnan(elev_coarse).sum()} / {elev_coarse.size}")

# Now zoom back up
z_r2 = H / coarse_shape[0]
z_c2 = W / coarse_shape[1]
e_ref = zoom(elev_coarse, (z_r2, z_c2), order=1)
print(f"\ne_ref at ({r},{c}): {e_ref[r,c]}")
print(f"e_ref NaN count: {np.isnan(e_ref).sum()} / {e_ref.size}")
print(f"\nelevation - e_ref at pixel: {elevation[r,c]} - {e_ref[r,c]} = {elevation[r,c] - e_ref[r,c]}")

Coarse temp shape: (9, 10)
30m elevation shape: (2488, 2593)

elev_coarse at approx location: nan
elev_coarse NaN count: 58 / 90

e_ref at (1337,2076): nan
e_ref NaN count: 4839160 / 6451384

elevation - e_ref at pixel: 2096.0 - nan = nan


In [ ]:
import os
RAW_LR = '/content/drive/MyDrive/02_NandiSeedRecommender2/Final_Outputs/Raw_Values_LongRains'
print("Files in Raw_Values_LongRains:")
for f in sorted(os.listdir(RAW_LR)):
    print(f" {f}")

Files in Raw_Values_LongRains:
 bedrock_depth_raw.tif
 calcium_raw.tif
 cec_apparent_raw.tif
 clay_content_raw.tif
 ecec_raw.tif
 germin_temp_raw.tif
 iron_raw.tif
 magnesium_raw.tif
 max_temp_raw.tif
 mean_season_rain_raw.tif
 mean_temp_raw.tif
 min_temp_raw.tif
 organic_carbon_raw.tif
 ph_raw.tif
 phosphorus_raw.tif
 potassium_raw.tif
 prec_month1_raw.tif
 prec_month2_raw.tif
 prec_month3_raw.tif
 prec_month4_raw.tif
 rh_dev_raw.tif
 rh_mat_raw.tif
 slope_raw.tif
 stone_content_raw.tif
 total_nitrogen_raw.tif
 zinc_raw.tif


###Pushing

In [ ]:
from google.colab import userdata
import os

USERNAME = "harryfyjiswalker"
REPO_OWNER = "Samarnorld"
REPO_NAME = "smartseed-backend"
TOKEN = userdata.get('GITHUB_TOKEN')

In [ ]:
!git add .

In [ ]:
!git commit -m "Add NandiSeedRecommender2 folder and new files"

On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean


In [ ]:
from google.colab import userdata
TOKEN = userdata.get('GITHUB_TOKEN')

!git pull https://{TOKEN}@github.com/Samarnorld/smartseed-backend.git main --rebase

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 6 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 1.78 KiB | 15.00 KiB/s, done.
From https://github.com/Samarnorld/smartseed-backend
 * branch            main       -> FETCH_HEAD
Successfully rebased and updated refs/heads/main.


In [ ]:
!git push https://{TOKEN}@github.com/Samarnorld/smartseed-backend.git main

Enumerating objects: 382, done.
Counting objects: 100% (382/382), done.
Delta compression using up to 2 threads
Compressing objects: 100% (381/381), done.
Writing objects: 100% (381/381), 975.71 MiB | 9.75 MiB/s, done.
Total 381 (delta 186), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (186/186), completed with 1 local object.
remote: warning: See https://gh.io/lfs for more information.
remote: warning: File 02_NandiSeedRecommender2/Final_Outputs/Risks_Mean_LongRains.tif is 66.84 MB; this is larger than GitHub's recommended maximum file size of 50.00 MB
remote: warning: File 02_NandiSeedRecommender2/Final_Outputs/Risks_Mean_ShortRains.tif is 66.19 MB; this is larger than GitHub's recommended maximum file size of 50.00 MB
remote: warning: File 02_NandiSeedRecommender2/Final_Outputs/Risks_Std_LongRains.tif is 67.74 MB; this is larger than GitHub's recommended maximum file size of 50.00 MB
remote: warning: File 02_NandiSeedRecommender2/Final_Outputs/Risks_Std_ShortRain